# Мультимодальная векторизация постов компаний $P^{(C)}$
## Модуль 1: Подготовка эмбеддингов для рекомендательной системы привлечения клиентов

Данный блокнот реализует пайплайн мультимодальной векторизации постов компаний на основе методов статьи:
- **Подключение к PostgreSQL (локально или на удаленном сервере)** с безопасным управлением секретами и SSL.
- **Частотно-временное сжатие видео по формуле MAD (Mean Absolute Difference)**:
  $$\text{MAD}(f_t, f_{t+1}) = \frac{1}{HW} \sum_{i=1}^H \sum_{j=1}^W |f_t(i, j) - f_{t+1}(i, j)|$$
  с отбором ключевых кадров $K = \{t \mid \text{MAD}(f_t, f_{t+1}) > \theta\}$, $|K| \le K_{\max}$.
- **Научное стратифицированное сэмплирование** на динамических квантилях генеральной совокупности.
- **Мультимодальный энкодер**: `Qwen3-VL-Embedding-2B` ($d_m = 2048$) с $L_2$-нормализацией.
- **Двухуровневая архитектура сохранения**: GPU микро-батчинг (32 поста) + транзакционный коммит в PostgreSQL каждые **1 000 постов**.
- **Диагностика латентного пространства**: Статистика Хопкинса ($H$), кумулятивная дисперсия PCA 95%, $\text{erank}$, проверка $L_2$-норм.

In [ ]:
# 1. Проверка окружения, оборудования и квоты VRAM (~20 ГБ H100)
import sys
from pathlib import Path

# Добавление src в PYTHONPATH
project_root = Path.cwd().resolve()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
elif (project_root.parent / "src").exists():
    sys.path.insert(0, str(project_root.parent / "src"))

print(f"Python: {sys.version}")
try:
    import torch

    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    vram_bytes = torch.cuda.get_device_properties(0).total_memory if cuda_available else 0
    total_vram_gb = vram_bytes / (1024**3)
    print(f"CUDA: {cuda_available} | Устройство: {device_name} | VRAM: {total_vram_gb:.2f} ГБ")
except ImportError:
    print("PyTorch не обнаружен. Доступен режим CPU / Mock.")

In [ ]:
# 2. Конфигурация запуска, подключение к PostgreSQL и безопасность секретов
from pathlib import Path

from vk_collector.ml.config import MLSettings
from vk_collector.ml.contracts import SampleMode
from vk_collector.ml.db_resolver import (
    build_ml_database_url,
    create_ml_database_engine,
    create_ml_session_factory,
)

# MLSettings автоматически читает параметры из .env или переменных окружения
settings = MLSettings(
    sample_mode=SampleMode.SME,  # MICRO (100), DEV (1000), SME (10000), LARGE (100000), FULL
    seed=42,
    allocated_gpu_vram_gb=20.0,  # Выделенная квота H100 из статьи (~20 GB)
    model_name="Qwen/Qwen3-VL-Embedding-2B",
    batch_size=32,  # GPU микро-батч (оптимально для 20 GB VRAM)
    db_save_interval=1000,  # Пакетное сохранение в PostgreSQL каждые 1 000 постов (~8.2 MB)
    export_dir=Path("exports/vectorization_runs"),
)

print(f"Режим сэмплирования: {settings.sample_mode.value}")
print(f"Модель: {settings.model_name} | GPU микро-батч: {settings.batch_size}")
print(f"Интервал коммита в PostgreSQL: {settings.db_save_interval} постов")

# Безопасный URL (пароль скрыт звездочками ***)
masked_url = build_ml_database_url(settings, mask_password=True)
raw_url = build_ml_database_url(settings, mask_password=False)
print(f"Строка подключения к БД: {masked_url}")

db_engine = create_ml_database_engine(
    raw_url,
    ssl_mode=settings.postgres_ssl_mode,
    ssl_ca_file=settings.postgres_ssl_ca_file,
)
db_session_factory = create_ml_session_factory(db_engine)

In [ ]:
# 3. Загрузка данных (из PostgreSQL или локального файла) и научное сэмплирование
from pathlib import Path

from vk_collector.ml.export_data import fetch_eligible_company_posts, load_posts_from_jsonl
from vk_collector.ml.sampling import generate_synthetic_posts, sample_company_posts

DATA_SOURCE = "offline"  # Варианты: 'database' (прямое чтение из PostgreSQL) или 'offline'

if DATA_SOURCE == "database":
    print("Чтение постов компаний из PostgreSQL (approved группы, 180 дней, <=100 на группу)...")
    async with db_session_factory() as session:
        population = await fetch_eligible_company_posts(
            session,
            window_days=settings.window_days,
            max_posts_per_group=settings.max_posts_per_group,
        )
    print(f"Извлечено {len(population)} постов из базы данных.")
else:
    data_path = Path("exports/company_posts_raw.jsonl")
    if data_path.exists():
        population = load_posts_from_jsonl(data_path)
        print(f"Загружено {len(population)} постов из {data_path}")
    else:
        print(f"Файл {data_path} не найден. Используется синтетическая популяция...")
        population = generate_synthetic_posts(n=12000)

# Выполнение многомерной стратификации
selected_posts, sampling_report = sample_company_posts(
    population,
    sample_mode=settings.sample_mode,
    seed=settings.seed,
)

print(f"Размер выборки: {len(selected_posts)} из {sampling_report.population_size}")
print(f"Покрытие страт (coverage): {sampling_report.strata_coverage:.1%}")
ks_d = sampling_report.ks_statistic_text_length
ks_p = sampling_report.ks_pvalue_text_length
print(f"KS-диагностика длины текста (D={ks_d:.4f}, p={ks_p:.4f})")

In [ ]:
# 4. Калибровка MAD и видеопроцессинг
from vk_collector.ml.video_calibration import VideoMADCalibrator
from vk_collector.ml.video_processor import VideoProcessor

# Калибровка на пилотной выборке видео
calibrator = VideoMADCalibrator(thetas=[0.05, 0.10, 0.15, 0.20], k_maxs=[4, 8, 12])
calib_result = calibrator.calibrate(["pilot_1.mp4", "pilot_2.mp4"])

theta_opt = calib_result.optimal_theta
k_opt = calib_result.optimal_k_max
print(f"Оптимальные MAD параметры: theta*={theta_opt}, K_max*={k_opt}")
sim_score = calib_result.mean_semantic_similarity
comp_ratio = calib_result.mean_compression_ratio
print(f"Сходство: {sim_score:.4f} | Сжатие: {comp_ratio:.2f}x")

video_proc = VideoProcessor(
    theta=calib_result.optimal_theta,
    k_max=calib_result.optimal_k_max,
    target_size=settings.target_frame_size,
)

In [ ]:
# 5. Инициализация энкодера и батчевая векторизация с коммитами по 1 000 постов
import uuid

from vk_collector.ml.dataset import CompanyPostsDataset
from vk_collector.ml.encoders.mock_encoder import MockMultimodalEncoder
from vk_collector.ml.runner import EmbeddingRunner

run_id = f"run_{uuid.uuid4().hex[:8]}"
dataset = CompanyPostsDataset(selected_posts, video_processor=video_proc)

# Адаптер энкодера (на GPU H100 используйте QwenVLEmbeddingAdapter)
encoder = MockMultimodalEncoder(
    model_name=settings.model_name,
    embedding_dim=settings.embedding_dim,
)

runner = EmbeddingRunner(
    encoder=encoder,
    batch_size=settings.batch_size,
    db_save_interval=settings.db_save_interval,  # Потоковая запись в PostgreSQL каждые 1 000 постов
    db_session_factory=db_session_factory if DATA_SOURCE == "database" else None,
    quota_gb=settings.allocated_gpu_vram_gb,
)

print(f"Запуск векторизации {len(dataset)} постов (run_id={run_id})...")
embeddings, succ_posts, failures = runner.run(
    dataset,
    run_id=run_id,
    resume=True,
    progress_callback=lambda cur, tot: print(f"Прогресс: {cur}/{tot}", end="\r"),
)

print(f"\nВекторизация завершена! Размер матрицы E^(C): {embeddings.shape}")
print(f"Успешно: {len(succ_posts)} | Сбоев: {len(failures)}")

In [ ]:
# 6. Диагностика качества эмбеддингов (Хопкинс, PCA 95%, erank, L2-норма)
from vk_collector.ml.metrics import evaluate_embedding_quality

quality_report = evaluate_embedding_quality(
    embeddings,
    run_id=run_id,
    model_name=settings.model_name,
    seed=settings.seed,
)

print("=== РЕЗУЛЬТАТЫ ДИАГНОСТИКИ ЛАТЕНТНОГО ПРОСТРАНСТВА ===")
h_stat = quality_report.hopkins_statistic
print(f"• Статистика Хопкинса (H): {h_stat:.4f} (H > 0.5: кластеризуемо)")
pca_comp = quality_report.pca_95_components
emb_d = quality_report.embedding_dim
print(f"• Компонент PCA для 95% дисперсии: {pca_comp} из {emb_d}")
print(f"• Эффективный ранг (erank): {quality_report.effective_rank:.2f}")
l2_status = "ВЫПОЛНЕНА" if quality_report.is_l2_normalized else "НАРУШЕНА"
print(f"• Строгая L2-нормализация: {l2_status}")
print(f"• NaN/Inf значения: {quality_report.nan_count} / {quality_report.inf_count}")
print(f"• Оценка анизотропии: {quality_report.anisotropy_score:.4f}")

In [ ]:
# 7. Визуализация латентного пространства (2D проекция PCA)
import numpy as np

mean_centered = embeddings - np.mean(embeddings, axis=0, keepdims=True)
_, _, vh = np.linalg.svd(mean_centered, full_matrices=False)
proj_2d = np.dot(mean_centered, vh[:2].T)

print(f"2D проекция рассчитана для {len(proj_2d)} точек.")
for i in range(min(5, len(proj_2d))):
    p = succ_posts[i]
    print(f"Пост {p.post_id} ({p.subject}): x={proj_2d[i, 0]:.3f}, y={proj_2d[i, 1]:.3f}")

In [ ]:
# 8. Экспорт самодостаточного бандла артефактов
from vk_collector.ml.artifacts import save_vectorization_bundle
from vk_collector.ml.contracts import ExecutionProvenance

provenance = ExecutionProvenance(
    run_id=run_id,
    seed=settings.seed,
    model_name=settings.model_name,
    config_hash="config_sha256_hash",
    dataset_hash="dataset_sha256_hash",
    mad_params=calib_result.model_dump(),
    allocated_gpu_vram_gb=settings.allocated_gpu_vram_gb,
)

run_output_dir = settings.export_dir / run_id
save_vectorization_bundle(
    run_output_dir,
    embeddings=embeddings,
    posts=succ_posts,
    provenance=provenance,
    sampling_report=sampling_report,
    quality_report=quality_report,
    failures=failures,
    calibration_result=calib_result,
)

print(f"Бандл артефактов сохранен в: {run_output_dir}")
print(f"Файлы: {[f.name for f in run_output_dir.iterdir()]}")